# 9주차 1팀

# 1. 자료불러오기





In [12]:
# 1) 내 Google Drive 연결 — 원시자료는 각자 자기 Drive 에 둡니다
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyreadstat
import pyreadstat
DATA_DIR = '/content/drive/MyDrive/hdac_data' # 팀 공통 경로

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# ==============================================================================
# Google Drive 연결 및 SAS 원시자료 로드 (실제 변수명 반영)
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyreadstat
import pyreadstat
import pandas as pd
import numpy as np

# 팀 공통 경로 및 파일명 설정
DATA_DIR = '/content/drive/MyDrive/원시자료'
FILE_NAME = 'kyrbs2025.sas7bdat'

# 분석에 사용할 실제 원변수명 목록
cols = [
    'M_SLP_HR_K',   # 주중 잠든 시각 (시간) - 노출 산출용
    'M_SLP_MM_K',   # 주중 잠든 시각 (분) - 노출 산출용
    'M_WK_HR_K',    # 주중 일어난 시각 (시간) - 노출 산출용
    'M_WK_MM_K',    # 주중 일어난 시각 (분) - 노출 산출용
    'E_S_RCRD',     # 주관적 학업성적 (결과)
    'SEX',          # 성별 (공변량)
    'GRADE',        # 학년 (공변량)
    'E_SES',        # 가구 경제 상태 (공변량)
    'M_STR',        # 평상시 스트레스 인지 (공변량)
    'PA_SWD_S',     # 주중 학습 목적 앉아서 보낸 시간 (공변량)
    'F_BR'          # 아침식사 횟수 등 (공변량)
]

print("SAS 원시자료를 불러오는 중입니다...")

# 파일 경로 조합 및 데이터 로드
file_path = f'{DATA_DIR}/{FILE_NAME}'
df, meta = pyreadstat.read_sas7bdat(file_path, usecols=cols)

print(f"[로드 완료] 전체 데이터 형태(행, 열): {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAS 원시자료를 불러오는 중입니다...
[로드 완료] 전체 데이터 형태(행, 열): (54170, 11)


# 2. 변수 확인 및 원 코드 빈도표

In [14]:
# =========================================================
# 2. 변수 확인 및 원 코드 빈도표
# =========================================================
print("=" * 60 + "\n[2] 변수 확인 및 원 코드 빈도표\n" + "=" * 60)

# 설계 변수 후보 자동 탐색 및 결합 (안전 장치)
WEIGHT_CANDS  = ['W', 'w']
STRATA_CANDS  = ['STRATA', 'strata']
CLUSTER_CANDS = ['CLUSTER', 'cluster', 'PSU', 'psu']

_, meta_all = pyreadstat.read_sas7bdat(file_path, metadataonly=True)
all_cols = list(meta_all.column_names)

def find_var(cands):
    return next((c for c in cands if c in all_cols), None)

W_VAR       = find_var(WEIGHT_CANDS)
STRATA_VAR  = find_var(STRATA_CANDS)
CLUSTER_VAR = find_var(CLUSTER_CANDS)
design_vars = [v for v in [W_VAR, STRATA_VAR, CLUSTER_VAR] if v]

if design_vars and not all(v in df.columns for v in design_vars):
    df_design, _ = pyreadstat.read_sas7bdat(file_path, usecols=design_vars)
    df_raw = pd.concat([df.reset_index(drop=True), df_design.reset_index(drop=True)], axis=1)
else:
    df_raw = df.copy()

print(f"원시자료 행 수: {len(df_raw):,}, 읽은 변수 수: {df_raw.shape[1]}")
print(f"설계변수 확인 -> 가중치={W_VAR}, 층={STRATA_VAR}, 집락={CLUSTER_VAR}")

summary = pd.DataFrame({
    "dtype": df_raw[df_raw.columns].dtypes.astype(str),
    "n_nan": df_raw[df_raw.columns].isna().sum(),
    "n_unique": df_raw[df_raw.columns].nunique(),
    "min": df_raw[df_raw.columns].min(numeric_only=True),
    "max": df_raw[df_raw.columns].max(numeric_only=True),
})
display(summary)

def freq_table(s):
    vc = s.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"n": vc, "%": (vc / vc.sum() * 100).round(2)})
    out.index.name = s.name
    return out

print("\n--- 주요 공변량 원 코드 빈도표 ---")
check_vars = ['E_S_RCRD', 'SEX', 'GRADE', 'E_SES', 'M_STR', 'F_BR']
for v in check_vars:
    if v in df_raw.columns:
        print(f"\n[변수: {v}]")
        display(freq_table(df_raw[v]))

print("\n[PA_SWD_S (주중 학습목적 앉아있는 시간 - 분 단위) 요약통계]")
display(df_raw['PA_SWD_S'].describe())

[2] 변수 확인 및 원 코드 빈도표
원시자료 행 수: 54,170, 읽은 변수 수: 14
설계변수 확인 -> 가중치=W, 층=STRATA, 집락=CLUSTER


,dtype,n_nan,n_unique,min,max
CLUSTER,float64,0,796,1.000000,800.0
E_SES,float64,12,5,1.000000,5.0
E_S_RCRD,float64,6,5,1.000000,5.0
F_BR,float64,0,8,1.000000,8.0
GRADE,float64,0,6,1.000000,6.0
M_SLP_HR_K,float64,0,24,1.000000,24.0
M_SLP_MM_K,float64,0,6,0.000000,50.0
M_STR,float64,0,5,1.000000,5.0
M_WK_HR_K,float64,0,24,1.000000,24.0
M_WK_MM_K,float64,0,6,0.000000,50.0



--- 주요 공변량 원 코드 빈도표 ---

[변수: E_S_RCRD]


,n,%
E_S_RCRD,,
1.0,6478,11.96
2.0,13870,25.60
3.0,15669,28.93
4.0,12865,23.75
5.0,5282,9.75
NaN,6,0.01



[변수: SEX]


,n,%
SEX,,
1.0,27761,51.25
2.0,26409,48.75


... (6 outputs left out)


[변수: F_BR]


,n,%
F_BR,,
1.0,14308,26.41
2.0,4236,7.82
3.0,5105,9.42
4.0,4153,7.67
5.0,3266,6.03
6.0,5859,10.82
7.0,3306,6.10
8.0,13937,25.73



[PA_SWD_S (주중 학습목적 앉아있는 시간 - 분 단위) 요약통계]


,PA_SWD_S
count,53467.000000
mean,432.751305
std,244.829807
min,0.000000
25%,240.000000
50%,450.000000
75%,600.000000
max,1438.000000


# 3. 무응답·비해당 코드 결측 처리 및 수면시간 산출

In [15]:
# =========================================================
# 3. 무응답·비해당 코드 결측 처리 및 수면시간 산출
# =========================================================
print("=" * 60)
print("[3] 무응답·비해당 코드 결측 처리 및 수면시간 산출")
print("=" * 60)

d = df_raw.copy()

# 1) 취침/기상 시각의 24시 응답을 0시(자정)로 변환
d.loc[d['M_SLP_HR_K'] == 24, 'M_SLP_HR_K'] = 0
d.loc[d['M_WK_HR_K'] == 24, 'M_WK_HR_K'] = 0

# 2) 이용지침서 기준 유효값 범위 설정 (PA_SWD_S 연속형 0~1438분 반영)
VALID = {
    'M_SLP_HR_K': range(0, 24),   # 0~23시 (24시는 0시로 변환됨)
    'M_SLP_MM_K': range(0, 60),
    'M_WK_HR_K':  range(0, 24),
    'M_WK_MM_K':  range(0, 60),
    'E_S_RCRD':   range(1, 6),    # 1(상) ~ 5(하)
    'SEX':        [1, 2],         # 1(남), 2(여)
    'GRADE':      range(1, 7),    # 1~3(중1~3), 4~6(고1~3)
    'E_SES':      range(1, 6),    # 1(상) ~ 5(하)
    'M_STR':      range(1, 6),
    'PA_SWD_S':   range(0, 1439), # 분 단위 연속형 (0~1,438분)
    'F_BR':       range(1, 9),    # 아침식사 횟수 (1~8)
}

# 유효범위를 벗어난 응답을 결측(NaN)으로 처리
for v, rng in VALID.items():
    if v in d.columns:
        d.loc[~d[v].isin(list(rng)), v] = np.nan

# 3) 수면시간 산출 (분 단위 -> 시간 단위 변환, 자정 넘김 보정)
bed_min = d['M_SLP_HR_K'] * 60 + d['M_SLP_MM_K']
wake_min = d['M_WK_HR_K'] * 60 + d['M_WK_MM_K']
sleep_min = np.where(wake_min < bed_min, wake_min + 1440 - bed_min, wake_min - bed_min)
d['SLEEP_HR'] = sleep_min / 60.0

# 4) 비현실적 수면시간 이상값 처리 (2시간 미만 또는 18시간 초과)
d.loc[(d['SLEEP_HR'] < 2.0) | (d['SLEEP_HR'] > 18.0), 'SLEEP_HR'] = np.nan

# 5) 노출군(EXPOSURE) 정의 (주 분석을 위한 2집단 체계로 확정)
# 0: 권장 수면충족군 (8~10시간, 기준군)
# 1: 수면 부족군 (< 8시간, 노출군)
# 10시간 초과 군은 주 분석 대상에서 제외(NaN 처리)하여 연구질문과 일치시킴
d['EXPOSURE'] = np.nan
d.loc[(d['SLEEP_HR'] >= 8.0) & (d['SLEEP_HR'] <= 10.0), 'EXPOSURE'] = 0
d.loc[d['SLEEP_HR'] < 8.0, 'EXPOSURE'] = 1

print("수면시간(SLEEP_HR) 요약통계:")
display(d['SLEEP_HR'].describe())

def freq_table(s):
    vc = s.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"n": vc, "%": (vc / vc.sum() * 100).round(2)})
    out.index.name = s.name
    return out

print()
print("확정된 노출군(EXPOSURE: 0=8~10시간, 1=<8시간) 분포 빈도:")
display(freq_table(d['EXPOSURE']))

[3] 무응답·비해당 코드 결측 처리 및 수면시간 산출
수면시간(SLEEP_HR) 요약통계:


,SLEEP_HR
count,50783.000000
mean,8.206434
std,2.024602
min,2.000000
25%,7.000000
50%,8.000000
75%,9.333333
max,18.000000



확정된 노출군(EXPOSURE: 0=8~10시간, 1=<8시간) 분포 빈도:


,n,%
EXPOSURE,,
0.0,23402,43.20
1.0,20741,38.29
NaN,10027,18.51


# 4. 대상 선정 단계별 인원 및 공변량 결측 분석

In [16]:
# =========================================================
# 4. 대상 선정 단계별 인원 및 공변량 결측 분석
# =========================================================
print("=" * 60)
print("[4] 대상 선정 단계별 인원 및 공변량 결측 분석")
print("=" * 60)

steps = []
def log_step(step_name, criterion, x):
    n_curr = len(x)
    n_prev = steps[-1]["남은 인원(명)"] if steps else len(d)
    steps.append({
        "단계": step_name,
        "기준": criterion,
        "남은 인원(명)": n_curr,
        "단계 제외(명)": n_prev - n_curr,
        "누적 제외율(%)": round((1 - n_curr / len(d)) * 100, 2),
        "가중합(추정모집단)": round(x[W_VAR].sum(), 0) if W_VAR and W_VAR in x.columns else None
    })

# 1단계: 전체 원시자료 응답자
s1 = d.copy()
log_step("1단계", "전체 원시자료 응답자", s1)

# 2단계: 모집단 필터링 (중1~고3 재학생)
s2 = s1[s1['GRADE'].between(1, 6)].copy()
log_step("2단계", "중1~고3 재학생 (GRADE 1~6)", s2)

# 3단계: 노출 변수(수면시간 8~10시간 또는 <8시간) 유효자 선정
s3 = s2.dropna(subset=['EXPOSURE']).copy()
log_step("3단계", "주중 수면시간 유효자 (권장충족군 및 부족군)", s3)

# 4단계: 결과 변수(학업성적) 결측 제외 -> 기본 분석 대상자
s4 = s3.dropna(subset=['E_S_RCRD']).copy()
log_step("4단계", "주관적 학업성적 (결과 변수) 유효 응답자 (기본 분석 대상)", s4)

flow_df = pd.DataFrame(steps)
display(flow_df)

COVARS = ['SEX', 'GRADE', 'E_SES', 'M_STR', 'PA_SWD_S', 'F_BR']
print()
print("[4단계 기본 분석 대상자(N = {:,}) 내 공변량별 결측 수]".format(len(s4)))
covar_missing = pd.DataFrame({
    "유효 응답자 수": s4[COVARS].notna().sum(),
    "결측자 수": s4[COVARS].isna().sum(),
    "결측 비율(%)": (s4[COVARS].isna().sum() / len(s4) * 100).round(2)
})
display(covar_missing)

# 5단계: 주요 공변량 결측 최종 제외 (완전 분석 대상)
s5 = s4.dropna(subset=COVARS).copy()
log_step("5단계", "주요 공변량 결측자 최종 제외 (완전 분석 대상)", s5)

df_final = s5.copy()
print()
print(f"최종 다중변수 분석 대상자 수: N = {len(df_final):,}명")
print("최종 분석 대상 노출군 분포:")
display(freq_table(df_final['EXPOSURE']))

[4] 대상 선정 단계별 인원 및 공변량 결측 분석


,단계,기준,남은 인원(명),단계 제외(명),누적 제외율(%),가중합(추정모집단)
0,1단계,전체 원시자료 응답자,54170,0,0.00,2642704.0
1,2단계,중1~고3 재학생 (GRADE 1~6),54170,0,0.00,2642704.0
2,3단계,주중 수면시간 유효자 (권장충족군 및 부족군),44143,10027,18.51,2166085.0
3,4단계,주관적 학업성적 (결과 변수) 유효 응답자 (기본 분석 대상),44138,5,18.52,2165847.0



[4단계 기본 분석 대상자(N = 44,138) 내 공변량별 결측 수]


,유효 응답자 수,결측자 수,결측 비율(%)
SEX,44138,0,0.00
GRADE,44138,0,0.00
E_SES,44134,4,0.01
M_STR,44138,0,0.00
PA_SWD_S,43565,573,1.30
F_BR,44138,0,0.00



최종 다중변수 분석 대상자 수: N = 43,562명
최종 분석 대상 노출군 분포:


,n,%
EXPOSURE,,
0.0,23098,53.02
1.0,20464,46.98
